# ViSta — a worked example on simulated observations

ViSta stacks interferometric observations of many faint galaxies **in the uv
plane** and measures the stacked line and continuum: profile, flux, width,
size and significance.

Here it stacks a **simulated sample**, twelve ALMA Band 3 observations of
galaxies at z = 2–3 whose CO(3–2) line no single galaxy detects, and then
changes **one option at a time** to show what each one does, against the
true values of the simulation.

Every comparison is one call, `vista.compare({...}, truth=...)`:

* top: the profile at zero spacing of every measurement (thin) and its fit
  (dashed);
* bottom left: the line flux per annulus of baseline length, with the source
  model (flat for a point source, falling for an extended one);
* bottom right: the fluxes with their errors (dot: the one quoted; square:
  the Gaussian; triangle: the area under the spectrum);
* green: the true values; below the figure, the numbers.

On Colab: *Runtime → Run all* (about 10 minutes on the CPU). Every
parameter is in the code cells: change it and run again from that cell.

## 0. Setup

The code is cloned from GitHub and the C++/OpenMP kernel (`ms_ops`) is built
with CMake (a couple of minutes, once). For a private repository, add a
GitHub token to the Colab secrets (key icon on the left) as `GITHUB_TOKEN`.
Outside Colab, set `VISTA_REPO` to a built checkout.

In [ ]:
REPO_URL = "https://github.com/martitors/ViSta-HPC.git"
BRANCH = "extract_opt"

import os, sys, glob, subprocess, logging
IN_COLAB = "google.colab" in sys.modules
REPO = os.environ.get("VISTA_REPO")            # a local checkout, to run this outside Colab
if REPO is None:
    REPO = "/content/ViSta-HPC" if IN_COLAB else os.path.abspath("ViSta-HPC")
    if not os.path.isdir(REPO):
        url = REPO_URL
        if IN_COLAB:
            try:                                 # private repository: token from the Colab secrets
                from google.colab import userdata
                url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
            except Exception:
                pass
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)

if IN_COLAB:
    subprocess.run(f"pip install -q -r {REPO}/requirements.txt pybind11", shell=True, check=True)
if not glob.glob(f"{REPO}/vista/ms_ops*.so"):          # build the kernel once
    subprocess.run(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release > /dev/null"
                   f" && cmake --build {REPO}/build -j 2 > /dev/null"
                   f" && cmake --install {REPO}/build > /dev/null", shell=True, check=True)
sys.path[:0] = [REPO, os.path.join(REPO, "examples")]

import simulate                                        # examples/simulate.py: the observations
import vista
from vista import ViSta

logging.getLogger("daskms").setLevel(logging.ERROR)    # quiet about empty optional columns
WORK = "/content/vista_demo" if IN_COLAB else os.path.abspath("vista_demo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("ViSta", vista.__version__, "from", REPO, "· working in", WORK)

## 1. A simulated sample

`simulate.make_sample` writes one Measurement Set per galaxy, as an ALMA-like
array (43 antennas within 600 m) would observe it in Band 3, and the input
list for ViSta. Every galaxy has a CO(3–2) line of 0.4 Jy km/s (at the median
redshift) and 300 km/s FWHM, a size of 0.8″ (resolved by the longest
baselines) and a dust continuum of 0.2 mJy, and sits up to 20″ from the
phase centre. As in real data, the datasets have different depths, their
weights misjudge the noise by 30 % (`kappa=1.3`) and adjacent channels are
correlated.

`simulate.expected` gives what the stack should measure: the green lines and
numbers of every comparison below.

In [ ]:
LINE = 345.79599                       # rest frequency of CO(3-2), GHz

truth = simulate.make_sample(
    "sim", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=300, size_arcsec=0.8,    # the line of every galaxy
    cont_flux=2e-4,                                   # the continuum, Jy
    noise=0.1, kappa=1.3, depth_spread_dex=0.2,       # noise, wrong weights, different depths
    offset_arcsec=lambda rng: tuple(rng.uniform(-15, 15, 2)),   # position in the field
    rmax=600, seed=1)
T = simulate.expected(truth)

print(f"true stack: line {T['flux'][0]:.3f} Jy km/s, continuum {T['continuum'] * 1e3:.2f} mJy "
      f"(at z_ref = {truth['z_ref']:.3f})\n")
print(open(truth["input_list"]).read())

## 2. Stack and measure

Three calls describe the run:

* `ViSta(...)`: the input list, the name of the outputs, how the run uses the
  machine;
* `v.stack(...)`: the line (its rest frequency and window), the band, the
  continuum (subtracted by default);
* `v.extract(...)`: the measurement, here with its defaults. Without this
  call, `run()` only writes the stack.

`run()` stacks the twelve datasets in the rest frame, subtracts the
continuum, compresses the stack into sums per channel and annulus of
baseline length, and measures the line on them (`ViSta(...,
verbose=True)` shows every step).

In [ ]:
v = ViSta(truth["input_list"], output_name="stack")
v.stack(line_freq=LINE, line_window=500, velocity_range=900)
v.extract()
r = v.run()
r

`r.plot()` draws the measurement. **Profile**: the stacked flux density at
zero spacing (black), the fitted Gaussian (red, its area hatched), the
window over which the spectrum is summed (orange), the noise (grey band) and
the imaginary part (dotted, pure noise); the box gives the flux quoted, the
two fluxes and the significance; the title, the source size and how much a
Gaussian source improves on a point source (Δχ²). **uv**: the line flux in
every annulus falls with the baseline length, because the galaxies are
extended; the fit extrapolates it to zero length.

In [ ]:
r.plot()

## 3. One option at a time

The stack and its statistics are written: every measurement below reads only
the statistics and takes seconds. `measure(...)` sets the options of
`extract()` and measures again; everything else stays as in §2. `v.help()`
lists every option, with its value now and what it can be.

In [ ]:
def measure(**options):
    """The measurement again with these options of extract(); the stack is not touched."""
    v.extract(**options)
    return v.run(only="measure", tag="variant")


v.help("extract")

## 4. Centre and width: searched or known

By default the centre and the width are searched within the window, and the
significance accounts for the search (the same search on noise). When they
are known, from another line or an optical redshift, fixing them
(`lo = hi`) turns the fit into a matched filter: a higher S/N and smaller
errors. A wrong assumed width biases the flux.

In [ ]:
vista.compare({
    "searched (default)": r,
    "centre and width known": measure(v0_range=(0, 0), fwhm_range=(300, 300)),
    "width assumed 200 km/s": measure(v0_range=(0, 0), fwhm_range=(200, 200)),
}, truth=T)

## 5. The source: point or extended

`model="auto"` (default) fits a point source and a Gaussian and averages them
with their weights. A point source misses the flux of an extended one: in the
uv panel the data fall with baseline, the flat model cannot follow them, and
the flux it extrapolates to zero length is low. A size fixed at the true
value is the best case; a range that excludes the true size
(`theta_range`) biases the flux too.

In [ ]:
vista.compare({
    "auto (default)": r,
    "point source": measure(model="point"),
    "Gaussian": measure(model="gauss"),
    "size fixed 0.8\"": measure(model=0.8),
    "size kept within 0.2-0.5\"": measure(theta_range=(0.2, 0.5)),
}, truth=T)

## 6. The window of the area flux

Two fluxes are always given: the integral of the fitted Gaussian, and the sum
of the spectrum over the window. A window narrower than the line loses part
of the area flux, not of the Gaussian one; `flux_measure="auto"` quotes the
Gaussian unless the two disagree and the window holds the whole line. The
window of the area flux can be changed without stacking again.

In [ ]:
vista.compare({
    "window ±500 km/s (default)": r,
    "window ±250 km/s": measure(window=250),
    "window ±120 km/s": measure(window=120),
}, truth=T)

## 7. Weighting, redshift and primary beam

* `weighting_scheme="democratic"`: every galaxy counts the same, instead of
  the deepest ones dominating (`"natural"`): the mean of the galaxies rather
  than of the data; here the same within the errors, which are larger.
* `z_norm=False`: the galaxies are not transported to the same redshift; the
  nearer ones are brighter, so the stack measures something else than the
  true value at `z_ref`.
* `pb_corr=False`: no correction for the primary beam: the galaxies, up to
  20″ from the phase centre, where the primary beam is 0.7–0.95, look fainter.

In [ ]:
vista.compare({
    "default": r,
    "democratic weighting": measure(weighting_scheme="democratic"),
    "no redshift rescaling": measure(z_norm=False),
    "no primary-beam correction": measure(pb_corr=False),
}, truth=T)

## 8. The errors

The noise of every channel and annulus is measured from the scatter of the
visibilities, and the correlation between adjacent channels from the data.
The figure of the noise: per channel, from the scatter and from the weights
(30 % lower here: the weights are wrong); the standardised noise against
N(0, 1); the search on noise that gives the significance.

In [ ]:
r.plot("noise")

Noise from the weights (which misjudge it here), or channels taken as
independent: the same fluxes, but errors too small and a significance far
too high.

In [ ]:
vista.compare({
    "default": r,
    "noise from the weights": measure(noise="weights"),
    "channels independent": measure(channel_correlation=False),
}, truth=T)

## 9. The continuum: subtracted, fitted or left

By default the continuum is subtracted while stacking, on the channels
outside the line window, and measured on its own. It can also be left in the
data and fitted together with the line (`continuum="fit"`), or left as it is
(`"none"`, for data already subtracted): here, with a continuum, the line
then sits on it and its flux is overestimated; the measurement flags the
continuum it finds outside the window. The stack keeps the data before the
subtraction, so this needs no new stack, only a new compression (seconds).

In [ ]:
continuum = {"subtracted (default)": r}
for mode, name in (("fit", "fitted with the line"), ("none", "left in the data")):
    v.stack(line_freq=LINE, line_window=500, velocity_range=900, continuum=mode)
    continuum[name] = measure()
v.stack(line_freq=LINE, line_window=500, velocity_range=900)     # back to the subtraction

vista.compare(continuum, truth=T)

The continuum measured on its own (with the subtraction): its flux density
in every annulus, and the fit of its size.

In [ ]:
r.plot("continuum")

## 10. A line that is not Gaussian

A second sample: the same kind of galaxies, but their line has broad wings
(a third of the flux in a component 800 km/s wide), and less noise. The
Gaussian fit follows the core and misses the wings; the sum over a window
wide enough catches them. `flux_measure="auto"` sees that the two disagree
and quotes the area.

In [ ]:
truth_w = simulate.make_sample(
    "wings", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=1,
    nchan=150,                                                   # a wider band
    extra_lines=[dict(rest_freq_ghz=LINE, flux_ratio=0.5, fwhm_kms=800)])   # the wings

w = ViSta(truth_w["input_list"], output_name="wings")
w.stack(line_freq=LINE, line_window=1000, velocity_range=1400)
w.extract(flux_measure="auto")
auto = w.run()                                                   # stack and measure
w.extract(flux_measure="gauss")
gauss = w.run(only="measure", tag="gauss")
w.extract(flux_measure="area")
area = w.run(only="measure", tag="area")

vista.compare({"Gaussian": gauss, "area": area, "auto": auto}, truth=simulate.expected(truth_w))

## 11. Two lines in the band

A third sample: every galaxy has a second line, half as bright, 400 km/s from
the first. With only the first line listed, the second sits partly in its
window (and its flux is added) and partly where the continuum is fitted.
Listed both, with windows that overlap, they are fitted together and
separated. Lines whose windows do not overlap are measured each on its own.

In [ ]:
LINE2 = LINE * (1 - 400 / 299792.458)          # 400 km/s from the first
truth_2 = simulate.make_sample(
    "two", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=2, nchan=150,
    extra_lines=[dict(rest_freq_ghz=LINE2, flux_ratio=0.5, fwhm_kms=250)])

d = ViSta(truth_2["input_list"], output_name="two")
d.extract()
d.stack(line_freq=LINE, line_window=300, velocity_range=1400)
first_only = d.run()
d.stack(line_freq=[LINE, LINE2], line_window=300, velocity_range=1400)
both = d.run()                                  # a new stack: the windows changed

vista.compare({"only the first listed": first_only, "both listed": both},
              truth=simulate.expected(truth_2))

## 12. The command line

`run_vista.sh` (or `python -m vista`) has the same options as `ViSta()`,
`stack()` and `extract()`, one flag each; `--extract` asks for the
measurement. It writes the same files; `--help` lists every option.

In [ ]:
!PYTHONPATH={REPO} python -m vista --input sim/input_list.txt --output-name cli/stack \
    --line-freq 345.796GHz --line-window 500 --velocity-range 900 \
    --extract --v0-range 0:0 --fwhm-range 300:300

## 13. On your own data

1. Write the input list, one line per dataset:
   `<ms_path> <redshift> <RA> <Dec> [FIELD_ID] [SPW_IDS] [NORM]`, with the
   position of the galaxy (hh:mm:ss.s ±dd:mm:ss.s).
2. `python examples/inspect_ms.py input_list.txt --line-freq 345.796GHz --all`
   reads the bands of the datasets and suggests `velocity_range`,
   `line_window` and `annuli`.
3. `ViSta(...)`, `v.stack(line_freq=..., line_window=...)` with a window wide
   enough for the whole line (one per line if the band holds more than one),
   `v.extract()`, `r = v.run()`, `r.plot()`.
4. Check `r.plot("noise")`, the continuum outside the window (on the
   profile), the significance, the size test and the two fluxes.
5. Change the options as above, with `run(only="measure")`, and compare the
   results with `vista.compare`.

The documentation, <https://vista-hpc.readthedocs.io>, explains every option.